# Evaluation: Lane 1 & Lane 2 

it imports the modules from the repo (`lane1.py`, `gm_convert.py`, `lane2.py`) and the reference models (`reference_models_v2.json`)

**Run order:** setup (clone repo + game_matching, put on path) → load models → Lane 1 → Lane 2.

> For a fully self-contained version that embeds everything see `evaluation.ipynb`. 

In [ ]:
# === setup (run once) ===
import os, sys, subprocess

def _clone(url, into):
    if not os.path.isdir(into):
        subprocess.run(["git", "clone", "-q", url, into], check=False)

# 1) game_matching (Lane 2 dependency)
_clone("https://github.com/mensch72/game_matching.git", "game_matching")

# 2) the evaluation modules. If running from a clone of world-model-suite, set
#    EVAL_DIR to the folder holding lane1.py (e.g. "json_generation"). Otherwise
#    we clone the suite and locate the modules inside it.
REPO_DIR = os.environ.get("EVAL_DIR", ".")
if not os.path.exists(os.path.join(REPO_DIR, "lane1.py")):
    _clone("https://github.com/GaganaMD/world-model-suite.git", "world-model-suite")
    for root, _, files in os.walk("world-model-suite"):
        if "lane1.py" in files:
            REPO_DIR = root
            break

sys.path.insert(0, "game_matching")
sys.path.insert(0, REPO_DIR)
print("using module dir:", REPO_DIR)

import lane1, gm_convert, lane2
from lane1 import run_lane1, print_lane1
from lane2 import lane2_score, print_lane2
print("imported lane1, gm_convert, lane2")

In [ ]:
# === load the reference models (source-of-truth JSON, not re-typed here) ===
import json, os
path = os.path.join(REPO_DIR, "reference_models.json")
with open(path) as fh:
    MODELS = json.load(fh)
print(len(MODELS), "reference models:", ", ".join(MODELS))

In [ ]:
# === Lane 1 on all reference models ===
print("LANE 1 mechanical validity\n" + "="*40)
l1 = {tc: run_lane1(m, f"TC {tc}") for tc, m in MODELS.items()}
for r in l1.values():
    print_lane1(r)
print(f"\n{sum(v['verdict']=='PASS' for v in l1.values())}/{len(MODELS)} pass Lane 1")

In [ ]:
# === Lane 2 on all reference models (renaming invariance) ===
import copy, random, string
def scramble(m, seed=7):
    rnd=random.Random(seed); lab=lambda:"".join(rnd.choice(string.ascii_lowercase) for _ in range(6))
    smap={s["id"]:lab() for s in m["states"]}; pmap={p["id"]:lab() for p in m["players"]}; amap={}
    for s in m["states"]:
        for e in s.get("actions",[]) or []:
            for a in e["by"].values(): amap.setdefault(a,lab())
    o=copy.deepcopy(m); o["players"]=[{**p,"id":pmap[p["id"]]} for p in o["players"]]
    o["initial"]=[{"state":smap[e["state"]],"p":e["p"]} for e in o["initial"]]
    ns=[]
    for s in o["states"]:
        s2=dict(s); s2["id"]=smap[s["id"]]
        if "turn" in s2: s2["turn"]=[pmap[p] for p in s2["turn"]]
        if "actions" in s2:
            s2["actions"]=[{"by":{pmap[p]:amap[a] for p,a in e["by"].items()},
                            "next":[{"state":smap[n["state"]],"p":n["p"]} for n in e["next"]]} for e in s2["actions"]]
        ns.append(s2)
    o["states"]=ns; return o

print("LANE 2 structural comparison up to renaming\n" + "="*46)
print(f"{'case':6}{'coverage':>10}{'precision':>11}{'lane2':>8}")
print("-"*35)
l2={}
for tc,m in MODELS.items():
    r=lane2_score(m, scramble(m), horizon=3, sa_iters=8000); l2[tc]=r
    print(f"{tc:6}{r['coverage']:>10.3f}{r['precision']:>11.3f}{r['lane2']:>8.3f}")
print(f"\n{sum(v['lane2']>0.999 for v in l2.values())}/{len(MODELS)} rename-invariant (score 1.000)")